# Simplex method benchmarks

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import sys
# sys.path.append('.')
# sys.path.append('./DynaMix/')

plt.rcParams["font.family"] = "Helvetica"

# Lyapunov exponent calculation

In [2]:
import sys
## add the path to the project
sys.path.append('..')

from models.parrot import SimplexForecaster, context_parroting_forecast


In [ ]:


# ?max_lyapunov_exponent_rosenstein
import sys
## add the path to the project
sys.path.append('..')

from models.parrot import SimplexForecaster


In [22]:
import torch
from chronos import BaseChronosPipeline, ChronosPipeline, ChronosBoltPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-base",  # use "amazon/chronos-bolt-small" for the corresponding Chronos-Bolt model
    #"amazon/chronos-bolt-base",
    #device_map="cuda",  # use "cpu" for CPU inference
    device_map="cpu",
    torch_dtype=torch.bfloat16,
)

In [ ]:
import sys
# sys.path.append('.')
sys.path.append('./DynaMix/')

import torch
# from src.model.dynamix import DynaMix
from src.model.forecaster import DynaMixForecaster
# from src.metrics.metrics import geometrical_misalignment, temporal_misalignment, MASE
# from src.utilities.plotting_eval import plot_3D_attractor, plot_2D_attractor, plot_TS_forecast
from src.utilities.utilities import load_hf_model

# Load the pre-trained model
model = load_hf_model("dynamix-3d-alrnn-v1.0")
model.eval() # Set model to evaluation mode
forecaster = DynaMixForecaster(model) # Initialize the forecaster

In [3]:
import sys
# sys.path.append('.')
sys.path.append('./DynaMix/')

import torch
# from src.model.dynamix import DynaMix
from src.model.forecaster import DynaMixForecaster
# from src.metrics.metrics import geometrical_misalignment, temporal_misalignment, MASE
# from src.utilities.plotting_eval import plot_3D_attractor, plot_2D_attractor, plot_TS_forecast
from src.utilities.utilities import load_hf_model

# Load the pre-trained model
model = load_hf_model("dynamix-3d-alrnn-v1.0")
model.eval() # Set model to evaluation mode
forecaster = DynaMixForecaster(model) # Initialize the forecaster

/Users/william/program_repos/parroting/benchmark/DynaMix/src/model/dynamix.py:95: RuntimeWarning: divide by zero encountered in matmul
  K = R.T @ R / M + np.eye(M)
/Users/william/program_repos/parroting/benchmark/DynaMix/src/model/dynamix.py:95: RuntimeWarning: overflow encountered in matmul
  K = R.T @ R / M + np.eye(M)
/Users/william/program_repos/parroting/benchmark/DynaMix/src/model/dynamix.py:95: RuntimeWarning: invalid value encountered in matmul
  K = R.T @ R / M + np.eye(M)


In [4]:
import glob
from dysts.analysis import max_lyapunov_exponent_rosenstein, max_lyapunov_exponent_rosenstein_multivariate, gp_dim
from dysts.metrics import estimate_kl_divergence


results_path = "../analysis/dynamix_statistics/"
CHECK_EXISTING = False

context_length = 512
forecast_length = 10000 - context_length


if CHECK_EXISTING:
    all_lyap = np.load(results_path + '/all_lyap_longhorizon.npy', allow_pickle=True).tolist()
    all_cdim = np.load(results_path + '/all_cdim_longhorizon.npy', allow_pickle=True).tolist()
    all_kl_dist = np.load(results_path + '/all_kl_dist_longhorizon.npy', allow_pickle=True).tolist()
else:
    all_lyap = list()
    all_cdim = list()
    all_kl_dist = list()

for trajectory in sorted(glob.glob("../data/long_trajectories/*.npy")):
    equation_name = trajectory.split("/")[-1].split(".")[0]
    print(equation_name, flush=True)
    # if equation_name in found_systems:
    #     print(f"Skipping {equation_name} because it's already in the found_systems list", flush=True)
    #     continue
    
    traj = np.load(trajectory, allow_pickle=True)
    traj = (traj - np.mean(traj, axis=0)) / np.std(traj, axis=0)

    try:

        traj_context = traj[:context_length, :]
        traj_true = traj[context_length:context_length+forecast_length, :]



        # traj_pred = list()
        # for mode in range(traj_context.shape[1]):
        #     traj_pred_mode = context_parroting_forecast(traj_context[:, mode], forecast_total_length=forecast_length)[2]
        #     traj_pred.append(traj_pred_mode)
        # traj_pred = np.array(traj_pred).T

        # traj_pred = list()
        # for mode in range(traj_context.shape[1]):
        #     model = SimplexForecaster()
        #     model.fit(traj_context[:, mode])
        #     traj_pred_mode = model.forecast(forecast_length)
        #     traj_pred.append(traj_pred_mode)
        # traj_pred = np.array(traj_pred).T


        # traj_pred = list()
        # for mode in range(traj_context.shape[1]):
        #     forecast = pipeline.predict(
        #         inputs=torch.tensor(traj_context[:, mode]),
        #         prediction_length=forecast_length,
        #         num_samples=20,
        #         limit_prediction_length=False,
        #         )
        #     traj_pred_mode = np.mean(forecast[0, :, :].detach().numpy(), axis=0)
        #     traj_pred.append(traj_pred_mode)
        # traj_pred = np.array(traj_pred).T


        context_traj_tensor  = torch.tensor(traj_context)
        with torch.no_grad(): 
            reconstruction = forecaster.forecast(
                context=context_traj_tensor,
                horizon=forecast_length, # Match the horizon of the ground truth
                standardize=True,
            )
        traj_pred = reconstruction.detach().numpy()

        
        kl_dist = estimate_kl_divergence(traj_true, traj_pred)
        if np.isinf(kl_dist): kl_dist = np.nan
        
        
        cdim_true = gp_dim(traj_true)
        cdim_pred = gp_dim(traj_pred)
        cdim = np.array([cdim_pred, cdim_true])

        lyap_true = max_lyapunov_exponent_rosenstein(traj_true)
        if np.isinf(lyap_true): lyap_true = 0
        lyap_pred = max_lyapunov_exponent_rosenstein(traj_pred)
        if np.isinf(lyap_pred): lyap_pred = 0
        lyap = np.array([lyap_pred, lyap_true])

        all_cdim.append(cdim)
        all_kl_dist.append(kl_dist)
        all_lyap.append(lyap)

        np.array(all_cdim).dump(results_path + '/all_cdim_longhorizon.npy')
        np.array(all_kl_dist).dump(results_path + '/all_kl_dist_longhorizon.npy')
        np.array(all_lyap).dump(results_path + '/all_lyap_longhorizon.npy')


    except Exception as e:
        print(e)
        print(f"Skipping {equation_name}", flush=True)
        continue


Aizawa


/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: divide by zero encountered in matmul
  return x @ self._LP
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: overflow encountered in matmul
  return x @ self._LP
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: invalid value encountered in matmul
  return x @ self._LP


AnishchenkoAstakhov
Arneodo
ArnoldBeltramiChildress
ArnoldWeb
AtmosphericRegime
cannot convert float NaN to integer
Skipping AtmosphericRegime
BeerRNN


/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


BelousovZhabotinsky
BickleyJet
Blasius
BlinkingRotlet
BlinkingVortex
Bouali
Bouali2
BurkeShaw
CaTwoPlus
CaTwoPlusQuasiperiodic
CellCycle
cannot convert float NaN to integer
Skipping CellCycle
CellularNeuralNetwork
Chen
ChenLee
Chua
CircadianRhythm
CoevolvingPredatorPrey
Colpitts
Coullet
Dadras
DequanLi
DoubleGyre
DoublePendulum
Duffing
ExcitableCell
Finance
ForcedBrusselator
ForcedFitzHughNagumo
ForcedVanDerPol
GenesioTesi
GlycolyticOscillation
GuckenheimerHolmes
Hadley
Halvorsen
HastingsPowell
HenonHeiles
HindmarshRose
Hopfield
HyperBao
HyperCai
HyperJha
HyperLorenz
HyperPang
HyperQi
HyperRossler
HyperWang
HyperXu
HyperYan
HyperYangChen
IkedaDelay
InteriorSquirmer
IsothermalChemical
ItikBanksTumor
JerkCircuit
KawczynskiStrizhak
Laser
LidDrivenCavityFlow
LiuChen
cannot convert float NaN to integer
Skipping LiuChen
Lorenz
Lorenz84
Lorenz96
LorenzBounded
LorenzCoupled
LorenzStenflo
LuChen
LuChenCheng
MacArthur
MackeyGlass
MooreSpiegel
MultiChua
NewtonLiepnik
NoseHoover
NuclearQuadrupole


In [3]:
# from scipy.stats import pearsonr, spearmanr

# model_names = ["parrot", "dynamix", "Chronos", "simplex"]
# # model_names = ["parrot", "dynamix",  "simplex"]

# from scipy.stats import norm
# def err_from_corr(corr, pvalue):
#     z_obs = np.arctanh(corr)
#     z_stat = abs(norm.ppf(pvalue / 2))
#     sd_z = abs(z_obs) / z_stat if z_stat != 0 else np.nan
#     sd_r = sd_z * (1 - corr**2)
#     return sd_r

# for model_name in model_names:
#     results_path = f"../analysis/{model_name}_statistics/"
#     all_lyap = np.load(results_path + '/all_lyap_longhorizon.npy', allow_pickle=True)
#     all_kl_dist = np.load(results_path + '/all_kl_dist_longhorizon.npy', allow_pickle=True)
#     all_cdim = np.load(results_path + '/all_cdim_longhorizon.npy', allow_pickle=True)

#     print(model_name)
#     # print(f"KL: {np.mean(all_kl_dist)}, {np.std(all_kl_dist) / np.sqrt(len(all_kl_dist))}")
#     ## string with 3 decimal places
#     print(f"KL: {np.mean(all_kl_dist):.3f} ± {np.std(all_kl_dist) / np.sqrt(len(all_kl_dist)):.3f}")
#     # print(pearsonr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[0], err_from_corr(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[0], spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[1]))
#     print(f"CDIM: {pearsonr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[0]:.3f} ± {err_from_corr(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[0], spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[1]):.3f}")
#     # print(pearsonr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[0], err_from_corr(spearmanr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[0], spearmanr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[1]))
#     print(f"LYAP: {pearsonr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[0]:.3f} ± {err_from_corr(spearmanr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[0], spearmanr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[1]):.3f}")
#     # print(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
#     # print(spearmanr(np.array(all_lyap)[:, 0], np.array(all_lyap)[:, 1]))
#     print("\n")

